# Agente Dinossauros — Algoritmo Genético

Agente evolutivo em ambiente simulado estilo Chrome Dino: população de k soluções, pressão seletiva e mutação, explorando o equilíbrio entre exploração e explotação sem gradiente explícito.

**Técnica:** Algoritmo genético / busca com múltiplos estados  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/09-dinossauros-algoritmo-genetico.ipynb)


In [1]:
"""
=============================================================================
AGENTE DINOSSAUROS COM ALGORITMO GENÉTICO
=============================================================================
Implementação de um Agente Evolutivo baseado em Algoritmos Genéticos (AG)
aplicado a um ambiente simulado estilo "Chrome Dino".

Fundamentos:
  - Busca estocástica: o AG é não-determinístico pois usa seleção probabilística,
    crossover em pontos aleatórios e mutação com probabilidade p. Dois runs com
    a mesma semente podem divergir se a semente não for fixada.
  - Busca com múltiplos estados: mantém uma POPULAÇÃO de k soluções simultâneas,
    diferente de hill-climbing (1 estado) ou busca em feixe determinística.
  - Exploração vs. Explotação: mutação explora regiões novas; seleção + crossover
    explota boas soluções já encontradas.
  - Evolução artificial: pressão seletiva + variação genética = melhoria iterativa
    sem gradiente explícito.
=============================================================================
"""

import random
import math
import copy
import statistics
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import time

# ---------------------------------------------------------------------------
# CONSTANTES GLOBAIS DO ALGORITMO GENÉTICO
# ---------------------------------------------------------------------------
TAMANHO_POPULACAO   = 60       # k indivíduos por geração
TAXA_CROSSOVER      = 0.85     # probabilidade de realizar crossover
TAXA_MUTACAO        = 0.12     # probabilidade de mutar cada gene
NUM_GERACOES        = 80       # critério de parada por iterações
ELITISMO            = 3        # top-N indivíduos preservados intactos
FITNESS_ALVO        = 5000     # critério de parada por fitness
TAMANHO_TORNEIO     = 4        # k candidatos no torneio de seleção
CROMOSSOMO_LEN      = 6        # número de genes por indivíduo

# ---------------------------------------------------------------------------
# DOMÍNIOS DOS GENES
# Cada gene[i] tem um intervalo [min, max] contínuo.
# Durante operações genéticas os valores são mantidos nesse domínio.
# ---------------------------------------------------------------------------
DOMINIOS = [
    (0.5,  2.5),   # gene[0] velocidade base       (unidades/tick)
    (0.05, 0.50),  # gene[1] tempo de reação        (fração do tick)
    (1.0,  4.0),   # gene[2] altura do pulo         (unidades)
    (0.1,  1.0),   # gene[3] frequência de decisão  (decisões/tick)
    (0.5,  3.0),   # gene[4] resistência/stamina    (multiplicador)
    (0.0,  1.0),   # gene[5] agressividade          (0=conservador, 1=agressivo)
]

NOMES_GENES = [
    "velocidade_base",
    "tempo_reacao",
    "altura_pulo",
    "freq_decisao",
    "resistencia",
    "agressividade",
]


# =============================================================================
# CLASSE: Individuo (Cromossomo)
# =============================================================================
class Individuo:
    """
    Representa um dinossauro como um cromossomo de valores reais.

    Cromossomo: vetor de CROMOSSOMO_LEN genes contínuos.
    Cada gene codifica um parâmetro comportamental do agente.

    Por que representação real e não binária?
      - Permite gradações finas nos parâmetros.
      - Facilita crossover aritmético e mutação gaussiana.
      - Mais natural para parâmetros físicos/comportamentais.
    """

    _id_counter = 0

    def __init__(self, genes: Optional[List[float]] = None):
        Individuo._id_counter += 1
        self.id = Individuo._id_counter
        self.genes: List[float] = genes if genes is not None else self._gerar_aleatorio()
        self.fitness: float = 0.0
        self.distancia: float = 0.0
        self.obstaculos_evitados: int = 0
        self.tempo_sobrevivido: int = 0

    # ------------------------------------------------------------------
    def _gerar_aleatorio(self) -> List[float]:
        """
        Gera um cromossomo com genes amostrados uniformemente nos domínios.
        A distribuição uniforme garante diversidade inicial máxima —
        nenhuma região do espaço de busca é favorecida a priori.
        """
        return [
            random.uniform(dom[0], dom[1])
            for dom in DOMINIOS
        ]

    # ------------------------------------------------------------------
    def clipar_genes(self):
        """Garante que todos os genes permaneçam dentro dos domínios válidos."""
        for i, (lo, hi) in enumerate(DOMINIOS):
            self.genes[i] = max(lo, min(hi, self.genes[i]))

    # ------------------------------------------------------------------
    def velocidade(self)    -> float: return self.genes[0]
    def reacao(self)        -> float: return self.genes[1]
    def altura_pulo(self)   -> float: return self.genes[2]
    def freq_decisao(self)  -> float: return self.genes[3]
    def resistencia(self)   -> float: return self.genes[4]
    def agressividade(self) -> float: return self.genes[5]

    # ------------------------------------------------------------------
    def __repr__(self) -> str:
        genes_str = ", ".join(f"{g:.3f}" for g in self.genes)
        return (f"Individuo(id={self.id}, fitness={self.fitness:.1f}, "
                f"genes=[{genes_str}])")


# =============================================================================
# CLASSE: Ambiente (Simulação)
# =============================================================================
class Ambiente:
    """
    Simula um cenário estilo 'Chrome Dino' de forma determinística dado
    um indivíduo. A aleatoriedade do ambiente é re-semeada por indivíduo
    para garantir comparabilidade justa entre todos na mesma geração.

    Mecânica:
      - O dinossauro corre da esquerda para a direita.
      - Obstáculos (cactos) aparecem em intervalos semi-aleatórios.
      - O agente decide pular ou não a cada tick com base em seus genes.
      - A simulação termina quando o dinossauro colide com um obstáculo.

    Por que um ambiente simulado?
      - Permite avaliação rápida e paralela de todos os indivíduos.
      - Controla a complexidade crescente ao longo das gerações.
    """

    LARGURA          = 200     # tamanho do "mundo"
    ALTURA_CHAO      = 0       # posição y do chão
    TICK_MAX         = 2000    # limite de ticks por avaliação
    GRAVIDADE        = 0.35    # aceleração de queda
    VELOCIDADE_MUNDO = 1.0     # velocidade base do scroll do mundo
    MIN_DIST_OBSTACULO = 20    # distância mínima entre obstáculos

    def __init__(self, seed: Optional[int] = None):
        self.seed = seed

    # ------------------------------------------------------------------
    def avaliar(self, individuo: Individuo) -> float:
        """
        Executa a simulação completa para um indivíduo e retorna o fitness.

        Retorna um valor escalar que representa a aptidão do indivíduo.
        Quanto maior, melhor — guia a pressão seletiva.
        """
        # Semente reprodutível por indivíduo garante fair comparison
        rng = random.Random(self.seed if self.seed else individuo.id % 9999)

        # Estado do dinossauro
        dino_y          = 0.0      # posição vertical (0 = chão)
        dino_vy         = 0.0      # velocidade vertical
        dino_no_chao    = True
        distancia       = 0.0
        obstaculos_evit = 0
        colidiu         = False

        # Estado do mundo
        obstaculos: List[dict] = []
        proximo_obstaculo      = rng.randint(15, 30)
        tick_proximo           = proximo_obstaculo

        # Velocidade crescente ao longo do tempo (dificuldade progressiva)
        velocidade_scroll = self.VELOCIDADE_MUNDO

        for tick in range(self.TICK_MAX):
            # ---- Aumenta dificuldade gradualmente ----
            if tick % 300 == 0 and tick > 0:
                velocidade_scroll += 0.08

            # ---- Geração de obstáculos ----
            tick_proximo -= 1
            if tick_proximo <= 0:
                altura_obs  = rng.uniform(1.0, 2.5)
                largura_obs = rng.uniform(1.0, 2.0)
                obstaculos.append({
                    "x":      self.LARGURA,
                    "altura": altura_obs,
                    "largura": largura_obs,
                })
                separacao = rng.randint(
                    self.MIN_DIST_OBSTACULO,
                    self.MIN_DIST_OBSTACULO + int(30 / max(velocidade_scroll, 0.5))
                )
                tick_proximo = separacao

            # ---- Movimento dos obstáculos ----
            for obs in obstaculos:
                obs["x"] -= velocidade_scroll * individuo.velocidade()

            # Remove obstáculos que saíram da tela
            obstaculos_antes = len(obstaculos)
            obstaculos = [o for o in obstaculos if o["x"] + o["largura"] > 0]
            obstaculos_evit += obstaculos_antes - len(obstaculos)

            # ---- Lógica de decisão do dinossauro ----
            # O agente decide pular com base nos genes e no obstáculo mais próximo
            if dino_no_chao and tick % max(1, int(1 / individuo.freq_decisao())) == 0:
                obstaculo_mais_proximo = self._obstaculo_mais_proximo(obstaculos)
                if obstaculo_mais_proximo is not None:
                    distancia_obs = obstaculo_mais_proximo["x"]
                    altura_obs    = obstaculo_mais_proximo["altura"]

                    # Limiar de pulo baseado em genes:
                    # Reação rápida = pula de longe; agressividade alta = pula mais cedo
                    limiar_pulo = (
                        5.0
                        + individuo.reacao() * 15.0
                        + individuo.agressividade() * 8.0
                        - individuo.velocidade() * 2.0
                    )
                    limiar_pulo = max(3.0, limiar_pulo)

                    deve_pular = (
                        distancia_obs < limiar_pulo
                        and distancia_obs > 0
                        and altura_obs > 0.5
                    )
                    if deve_pular:
                        dino_vy     = individuo.altura_pulo() * 1.8
                        dino_no_chao = False

            # ---- Física do pulo ----
            if not dino_no_chao:
                dino_y  += dino_vy
                dino_vy -= self.GRAVIDADE
                if dino_y <= self.ALTURA_CHAO:
                    dino_y       = 0.0
                    dino_vy      = 0.0
                    dino_no_chao = True

            # ---- Detecção de colisão ----
            DINO_X      = 10      # posição x fixa do dinossauro
            DINO_ALTURA = 2.0
            DINO_LARGURA = 1.0

            for obs in obstaculos:
                colisao_x = (DINO_X + DINO_LARGURA > obs["x"] and
                             DINO_X < obs["x"] + obs["largura"])
                colisao_y = (dino_y < obs["altura"] and
                             dino_y + DINO_ALTURA > 0)
                if colisao_x and colisao_y:
                    colidiu = True
                    break

            if colidiu:
                break

            distancia += velocidade_scroll * individuo.velocidade()

        # ---- Calcula fitness ----
        fitness = self._calcular_fitness(
            distancia=distancia,
            obstaculos_evitados=obstaculos_evit,
            tempo=tick,
            colidiu=colidiu,
            individuo=individuo,
        )

        # Armazena métricas no indivíduo
        individuo.distancia          = distancia
        individuo.obstaculos_evitados = obstaculos_evit
        individuo.tempo_sobrevivido  = tick
        individuo.fitness            = fitness

        return fitness

    # ------------------------------------------------------------------
    @staticmethod
    def _obstaculo_mais_proximo(obstaculos: List[dict]) -> Optional[dict]:
        """Retorna o obstáculo à frente mais próximo do dinossauro."""
        a_frente = [o for o in obstaculos if o["x"] > 8]
        if not a_frente:
            return None
        return min(a_frente, key=lambda o: o["x"])

    # ------------------------------------------------------------------
    @staticmethod
    def _calcular_fitness(
        distancia: float,
        obstaculos_evitados: int,
        tempo: int,
        colidiu: bool,
        individuo: Individuo,
    ) -> float:
        """
        Função de Fitness — coração do AG.

        f(x) = distância_percorrida
               + bônus_obstáculos
               + bônus_tempo
               + bônus_sobrevivência
               - penalidade_colisão

        Por que esta função?
          - Distância é proxy de desempenho geral.
          - Bônus por obstáculo incentiva decisões corretas específicas.
          - Penalidade por colisão cria pressão seletiva contra falhas graves.
          - Bônus de resistência valoriza genes que sustentam longa corrida.

        Possíveis vieses:
          - Se bônus de obstáculos for muito alto, o agente pode aprender a
            pular sempre, mesmo sem obstáculos (falso positivo).
          - Solução: calibrar pesos cuidadosamente.
        """
        fitness  = distancia * 1.0
        fitness += obstaculos_evitados * 20.0
        fitness += tempo * 0.5
        fitness += individuo.resistencia() * 50.0  # genes intrinsecamente bons

        if not colidiu:
            fitness += 500.0  # bônus de sobrevivência completa

        return max(0.0, fitness)


# =============================================================================
# MÓDULO: Operadores Genéticos
# =============================================================================

def crossover_ponto_unico(pai1: Individuo, pai2: Individuo) -> Tuple[Individuo, Individuo]:
    """
    Crossover de ponto único (Single-Point Crossover).

    Escolhe um ponto de corte k aleatório em [1, L-1].
    Filho1 = pai1[:k] + pai2[k:]
    Filho2 = pai2[:k] + pai1[k:]

    Teoria dos Building Blocks (Holland, 1975):
      - Blocos de genes co-adaptados tendem a permanecer juntos se o ponto
        de corte cair fora deles.
      - Favorece a recombinação de sub-soluções parcialmente boas.
    """
    if random.random() > TAXA_CROSSOVER or len(pai1.genes) < 2:
        return copy.deepcopy(pai1), copy.deepcopy(pai2)

    k = random.randint(1, len(pai1.genes) - 1)
    genes1 = pai1.genes[:k] + pai2.genes[k:]
    genes2 = pai2.genes[:k] + pai1.genes[k:]

    filho1 = Individuo(genes=genes1)
    filho2 = Individuo(genes=genes2)
    return filho1, filho2


def crossover_aritmetico(pai1: Individuo, pai2: Individuo, alpha: float = 0.5) -> Tuple[Individuo, Individuo]:
    """
    Crossover Aritmético (BLX-alpha / interpolação linear).

    Para cada gene i:
      alpha aleatório em [0, 1]
      filho1[i] = alpha * pai1[i] + (1-alpha) * pai2[i]
      filho2[i] = (1-alpha) * pai1[i] + alpha * pai2[i]

    Vantagem: gera filhos dentro do convex hull dos pais → suave.
    Desvantagem: pode reduzir diversidade se pais forem próximos.
    """
    if random.random() > TAXA_CROSSOVER:
        return copy.deepcopy(pai1), copy.deepcopy(pai2)

    a = random.random()
    genes1 = [a * g1 + (1-a) * g2 for g1, g2 in zip(pai1.genes, pai2.genes)]
    genes2 = [(1-a) * g1 + a * g2 for g1, g2 in zip(pai1.genes, pai2.genes)]

    filho1 = Individuo(genes=genes1)
    filho2 = Individuo(genes=genes2)
    filho1.clipar_genes()
    filho2.clipar_genes()
    return filho1, filho2


def mutar(individuo: Individuo, taxa: float = TAXA_MUTACAO) -> Individuo:
    """
    Mutação Gaussiana por gene.

    Para cada gene i com probabilidade `taxa`:
      gene[i] += N(0, sigma_i)
    onde sigma_i é proporcional à amplitude do domínio.

    Importância da Mutação:
      - Previne convergência prematura (população presa em ótimo local).
      - Reintroduz diversidade perdida por deriva genética.
      - Permite explorar regiões do espaço não alcançadas por crossover.

    Mutação gaussiana é preferível à mutação uniforme para representações
    reais pois introduz perturbações pequenas com maior probabilidade —
    consistente com o princípio de que pequenas mudanças são mais prováveis
    de melhorar uma solução já boa.
    """
    novo = copy.deepcopy(individuo)
    for i, (lo, hi) in enumerate(DOMINIOS):
        if random.random() < taxa:
            sigma = (hi - lo) * 0.15   # 15% da amplitude como desvio padrão
            novo.genes[i] += random.gauss(0, sigma)
    novo.clipar_genes()
    novo.id = Individuo._id_counter + 1
    Individuo._id_counter += 1
    return novo


def selecao_torneio(populacao: List[Individuo], k: int = TAMANHO_TORNEIO) -> Individuo:
    """
    Seleção por Torneio.

    Amostra k indivíduos aleatórios e retorna o de maior fitness.

    Vantagens sobre roleta:
      - Menos sensível a outliers de fitness muito alto (sem dominância).
      - Complexidade O(k) em vez de O(n).
      - Pressão seletiva controlada pelo parâmetro k.
        k=2 → seleção fraca; k=n → seleção forte (equivale a greedy).

    Por que seleção estocástica?
      - Permite que indivíduos medianos ocasionalmente sejam selecionados,
        mantendo diversidade e evitando convergência prematura.
      - Diferencia o AG de um algoritmo guloso determinístico.
    """
    candidatos = random.sample(populacao, min(k, len(populacao)))
    return max(candidatos, key=lambda ind: ind.fitness)


def selecao_roleta(populacao: List[Individuo]) -> Individuo:
    """
    Seleção por Roleta (Fitness-Proportionate Selection).

    P(indivíduo i) = fitness_i / sum(fitness_j para j em população)

    Problema: se um indivíduo tiver fitness muito superior, domina as
    seleções (super-indivíduo) → convergência prematura.
    Solução alternativa: usar torneio (implementado acima).
    """
    total = sum(ind.fitness for ind in populacao)
    if total == 0:
        return random.choice(populacao)
    r = random.uniform(0, total)
    acumulado = 0.0
    for ind in populacao:
        acumulado += ind.fitness
        if acumulado >= r:
            return ind
    return populacao[-1]


# =============================================================================
# CLASSE: Populacao
# =============================================================================
class Populacao:
    """
    Representa o conjunto de indivíduos em uma geração.

    Diversidade genética inicial:
      - Garante que a busca não comece em um único ponto do espaço.
      - Populações pequenas convergem mais rápido mas arriscam ótimos locais.
      - Populações grandes exploram melhor mas são mais custosas.

    Elitismo:
      - Preserva os N melhores indivíduos sem modificação.
      - Garante que o melhor fitness nunca decresce entre gerações.
      - Contrapõe o risco de mutação destruir boas soluções.
    """

    def __init__(self, tamanho: int = TAMANHO_POPULACAO):
        self.tamanho    = tamanho
        self.individuos: List[Individuo] = [Individuo() for _ in range(tamanho)]
        self.geracao    = 0
        self.historico_melhor: List[float] = []
        self.historico_media:  List[float] = []
        self.historico_pior:   List[float] = []

    # ------------------------------------------------------------------
    def ordenar(self):
        self.individuos.sort(key=lambda x: x.fitness, reverse=True)

    def melhor(self)  -> Individuo: return self.individuos[0]
    def pior(self)    -> Individuo: return self.individuos[-1]

    def fitness_medio(self) -> float:
        if not self.individuos:
            return 0.0
        return statistics.mean(ind.fitness for ind in self.individuos)

    def fitness_desvio(self) -> float:
        if len(self.individuos) < 2:
            return 0.0
        return statistics.stdev(ind.fitness for ind in self.individuos)

    # ------------------------------------------------------------------
    def registrar_geracao(self):
        self.historico_melhor.append(self.melhor().fitness)
        self.historico_media.append(self.fitness_medio())
        self.historico_pior.append(self.pior().fitness)

    # ------------------------------------------------------------------
    def elite(self, n: int = ELITISMO) -> List[Individuo]:
        """Retorna cópias profundas dos N melhores indivíduos (elitismo)."""
        return [copy.deepcopy(ind) for ind in self.individuos[:n]]


# =============================================================================
# CLASSE: AlgoritmoGenetico
# =============================================================================
class AlgoritmoGenetico:
    """
    Orquestrador principal do ciclo evolutivo.

    Ciclo por geração:
      1. Avaliar toda a população no ambiente.
      2. Ordenar por fitness.
      3. Registrar métricas.
      4. Exibir progresso.
      5. Verificar critério de parada.
      6. Gerar nova população:
         a. Preservar elite.
         b. Selecionar pais por torneio.
         c. Aplicar crossover.
         d. Aplicar mutação.
      7. Substituir população.
      8. Incrementar geração.

    Por que AG é busca estocástica?
      - Seleção: amostragem probabilística (torneio aleatório).
      - Crossover: ponto de corte aleatório.
      - Mutação: perturbação gaussiana aleatória por gene.
      - Resultado: cada execução pode produzir trajetória diferente.
      Diferença de busca determinística: não há função de gradiente,
      nenhuma garantia de convergência global, mas boa performance média.

    Como trabalha com múltiplos estados:
      - Mantém k=TAMANHO_POPULACAO soluções candidatas simultaneamente.
      - Cada geração, a população "migra" coletivamente em direção a
        regiões de maior fitness.
      - Análogo a k agentes independentes que trocam informação genética.
    """

    def __init__(
        self,
        tamanho_pop: int       = TAMANHO_POPULACAO,
        num_geracoes: int      = NUM_GERACOES,
        taxa_crossover: float  = TAXA_CROSSOVER,
        taxa_mutacao: float    = TAXA_MUTACAO,
        elitismo: int          = ELITISMO,
        fitness_alvo: float    = FITNESS_ALVO,
        verbose: bool          = True,
        usar_crossover_arit: bool = False,
    ):
        self.tamanho_pop        = tamanho_pop
        self.num_geracoes       = num_geracoes
        self.taxa_crossover     = taxa_crossover
        self.taxa_mutacao       = taxa_mutacao
        self.elitismo_n         = elitismo
        self.fitness_alvo       = fitness_alvo
        self.verbose            = verbose
        self.usar_crossover_arit = usar_crossover_arit

        self.populacao   = Populacao(tamanho_pop)
        self.ambiente    = Ambiente(seed=42)
        self.melhor_ever: Optional[Individuo] = None
        self.inicio      = time.time()

    # ------------------------------------------------------------------
    def _avaliar_populacao(self):
        """Avalia todos os indivíduos e atualiza seus fitness."""
        for ind in self.populacao.individuos:
            self.ambiente.avaliar(ind)

    # ------------------------------------------------------------------
    def _gerar_nova_populacao(self) -> List[Individuo]:
        """
        Gera a próxima geração via:
          1. Elitismo: preserva os N melhores sem alteração.
          2. Para o restante: seleciona pais, aplica crossover e mutação.

        Elitismo garante que o melhor fitness seja monotonicamente
        não-decrescente ao longo das gerações.
        """
        nova_pop: List[Individuo] = []

        # 1. Elitismo
        nova_pop.extend(self.populacao.elite(self.elitismo_n))

        # 2. Preenchimento por seleção + crossover + mutação
        while len(nova_pop) < self.tamanho_pop:
            pai1 = selecao_torneio(self.populacao.individuos)
            pai2 = selecao_torneio(self.populacao.individuos)

            if self.usar_crossover_arit:
                filho1, filho2 = crossover_aritmetico(pai1, pai2)
            else:
                filho1, filho2 = crossover_ponto_unico(pai1, pai2)

            filho1 = mutar(filho1, self.taxa_mutacao)
            filho2 = mutar(filho2, self.taxa_mutacao)

            nova_pop.append(filho1)
            if len(nova_pop) < self.tamanho_pop:
                nova_pop.append(filho2)

        return nova_pop[:self.tamanho_pop]

    # ------------------------------------------------------------------
    def _atualizar_melhor_ever(self):
        """Rastreia o melhor indivíduo de todas as gerações (Hall of Fame)."""
        atual_melhor = self.populacao.melhor()
        if self.melhor_ever is None or atual_melhor.fitness > self.melhor_ever.fitness:
            self.melhor_ever = copy.deepcopy(atual_melhor)

    # ------------------------------------------------------------------
    def _exibir_geracao(self):
        """Exibe o progresso da geração atual de forma formatada."""
        g     = self.populacao.geracao
        melhor = self.populacao.melhor()
        media  = self.populacao.fitness_medio()
        desvio = self.populacao.fitness_desvio()
        pior   = self.populacao.pior()
        elapsed = time.time() - self.inicio

        separador = "─" * 60
        print(f"\n{separador}")
        print(f"  GERAÇÃO {g:>3d}   |   Tempo: {elapsed:.1f}s")
        print(separador)
        print(f"  Melhor fitness : {melhor.fitness:>10.1f}   "
              f"(dist={melhor.distancia:.0f}, obs={melhor.obstaculos_evitados})")
        print(f"  Média          : {media:>10.1f}   ±{desvio:.1f}")
        print(f"  Pior           : {pior.fitness:>10.1f}")
        print(f"  Melhor all-time: {self.melhor_ever.fitness:>10.1f}")
        print(f"  Genes do melhor: {[f'{g:.3f}' for g in melhor.genes]}")
        print(separador)

    # ------------------------------------------------------------------
    def _exibir_progresso_visual(self):
        """Exibe barra de progresso ASCII do fitness relativo ao alvo."""
        melhor_fit = self.populacao.melhor().fitness
        proporcao  = min(1.0, melhor_fit / self.fitness_alvo)
        largura    = 40
        preenchido = int(proporcao * largura)
        barra      = "█" * preenchido + "░" * (largura - preenchido)
        print(f"  Progresso ao alvo: [{barra}] {proporcao*100:.1f}%")

    # ------------------------------------------------------------------
    def executar(self) -> Individuo:
        """
        Loop evolutivo principal.

        Critérios de parada:
          1. Número máximo de gerações atingido.
          2. Fitness alvo alcançado (convergência suficiente).

        Retorna o melhor indivíduo encontrado em todas as gerações.
        """
        print("=" * 60)
        print("  ALGORITMO GENÉTICO — AGENTE DINOSSAUROS")
        print("=" * 60)
        print(f"  População     : {self.tamanho_pop} indivíduos")
        print(f"  Gerações      : {self.num_geracoes}")
        print(f"  Taxa crossover: {self.taxa_crossover}")
        print(f"  Taxa mutação  : {self.taxa_mutacao}")
        print(f"  Elitismo      : {self.elitismo_n}")
        print(f"  Fitness alvo  : {self.fitness_alvo}")
        print(f"  Crossover     : {'Aritmético' if self.usar_crossover_arit else 'Ponto único'}")
        print("=" * 60)

        for geracao in range(1, self.num_geracoes + 1):
            self.populacao.geracao = geracao

            # 1. Avaliação
            self._avaliar_populacao()

            # 2. Ordenação
            self.populacao.ordenar()

            # 3. Registro histórico
            self.populacao.registrar_geracao()

            # 4. Rastrear melhor global
            self._atualizar_melhor_ever()

            # 5. Exibição
            if self.verbose:
                self._exibir_geracao()
                self._exibir_progresso_visual()

            # 6. Critério de parada por fitness
            if self.melhor_ever.fitness >= self.fitness_alvo:
                print(f"\n  ✔ FITNESS ALVO ATINGIDO na geração {geracao}!")
                break

            # 7. Nova geração (exceto na última iteração)
            if geracao < self.num_geracoes:
                nova_pop = self._gerar_nova_populacao()
                self.populacao.individuos = nova_pop

        self._exibir_resultado_final()
        return self.melhor_ever

    # ------------------------------------------------------------------
    def _exibir_resultado_final(self):
        """Exibe resumo completo da evolução e o melhor indivíduo."""
        print("\n" + "=" * 60)
        print("  RESULTADO FINAL")
        print("=" * 60)

        ind = self.melhor_ever
        print(f"\n  Melhor dinossauro encontrado (ID={ind.id}):")
        print(f"  Fitness          : {ind.fitness:.2f}")
        print(f"  Distância        : {ind.distancia:.1f}")
        print(f"  Obstáculos evit. : {ind.obstaculos_evitados}")
        print(f"  Tempo sobrevivido: {ind.tempo_sobrevivido} ticks")

        print("\n  Genoma (cromossomo):")
        for nome, valor, (lo, hi) in zip(NOMES_GENES, ind.genes, DOMINIOS):
            barra_len = 20
            proporcao = (valor - lo) / (hi - lo)
            barra = "█" * int(proporcao * barra_len) + "░" * (barra_len - int(proporcao * barra_len))
            print(f"  {nome:<20s}: {valor:.4f}  [{barra}]  (dom: [{lo}, {hi}])")

        print("\n  Evolução do Fitness por Geração:")
        self._exibir_grafico_ascii()
        print("=" * 60)

    # ------------------------------------------------------------------
    def _exibir_grafico_ascii(self):
        """
        Exibe gráfico ASCII da evolução do fitness ao longo das gerações.
        Mostra melhor (█), média (▓) e pior (░) fitness por geração.
        """
        historico = self.populacao.historico_melhor
        if not historico:
            return

        altura    = 12
        largura   = min(60, len(historico))
        maximo    = max(historico) if historico else 1
        minimo    = min(self.populacao.historico_pior) if self.populacao.historico_pior else 0
        amplitude = max(maximo - minimo, 1)

        # Subamostrar se tiver muitas gerações
        step = max(1, len(historico) // largura)
        melhores = historico[::step][:largura]
        medias   = self.populacao.historico_media[::step][:largura]

        print(f"\n  Fitness (máx={maximo:.0f})")
        print(f"  {'▲':>3}")

        for linha in range(altura, 0, -1):
            limiar = minimo + (linha / altura) * amplitude
            row = ""
            for i, (m, med) in enumerate(zip(melhores, medias)):
                if m >= limiar:
                    row += "█"
                elif med >= limiar:
                    row += "▒"
                else:
                    row += " "
            label = f"{limiar:>7.0f} |" if linha % 3 == 0 else "        |"
            print(f"  {label}{row}")

        print(f"  {'0':>7} +" + "─" * largura)
        print(f"  {'':>8} Geração 1" + " " * (largura - 18) + f"Geração {len(historico)}")
        print(f"\n  Legenda: █=melhor  ▒=média")


# =============================================================================
# MÓDULO: Análise e Diagnóstico
# =============================================================================

def analisar_convergencia(ag: AlgoritmoGenetico):
    """
    Analisa a convergência da população ao longo das gerações.

    Diagnósticos:
      - Diversidade: desvio padrão do fitness. Valor baixo → convergência.
      - Melhoria relativa: taxa de crescimento do melhor fitness.
      - Geração de estagnação: quantas gerações sem melhoria significativa.
    """
    print("\n" + "=" * 60)
    print("  ANÁLISE DE CONVERGÊNCIA")
    print("=" * 60)

    historico = ag.populacao.historico_melhor
    if len(historico) < 2:
        print("  Dados insuficientes para análise.")
        return

    # Melhoria total
    melhoria_total = historico[-1] - historico[0]
    print(f"  Melhoria total    : {melhoria_total:+.1f} "
          f"({100*melhoria_total/max(historico[0],1):.1f}% de crescimento)")

    # Geração de maior salto
    saltos = [historico[i+1] - historico[i] for i in range(len(historico)-1)]
    gen_salto = saltos.index(max(saltos)) + 1
    print(f"  Maior salto       : geração {gen_salto} (+{max(saltos):.1f})")

    # Estagnação
    threshold = 1.0
    estagnacao = 0
    for s in reversed(saltos):
        if abs(s) < threshold:
            estagnacao += 1
        else:
            break
    print(f"  Gerações estagnado: {estagnacao} últimas gerações")

    # Velocidade de convergência (geração onde atingiu 90% do máximo)
    alvo_90 = 0.9 * historico[-1]
    gen_90  = next((i+1 for i, f in enumerate(historico) if f >= alvo_90), len(historico))
    print(f"  90% do máximo     : atingido na geração {gen_90}")

    print("=" * 60)


def comparar_individuos(ind1: Individuo, ind2: Individuo):
    """Exibe comparação lado a lado de dois indivíduos."""
    print("\n  COMPARAÇÃO DE INDIVÍDUOS")
    print(f"  {'Gene':<20} {'Ind A':>10} {'Ind B':>10} {'Vencedor':>10}")
    print("  " + "─" * 52)
    for nome, v1, v2 in zip(NOMES_GENES, ind1.genes, ind2.genes):
        vencedor = "A" if v1 > v2 else ("B" if v2 > v1 else "Empate")
        print(f"  {nome:<20} {v1:>10.4f} {v2:>10.4f} {vencedor:>10}")
    print(f"  {'FITNESS':<20} {ind1.fitness:>10.1f} {ind2.fitness:>10.1f}")


# =============================================================================
# EXPERIMENTO: Múltiplos Runs (análise de variância)
# =============================================================================

def executar_multiplos_runs(n_runs: int = 3, geracoes: int = 30, verbose_runs: bool = False):
    """
    Executa N runs independentes e compara resultados.

    Importância:
      - AG é estocástico → resultados variam entre runs.
      - Análise de variância revela robustez do algoritmo.
      - Média e desvio padrão dos melhores fitness medem estabilidade.
    """
    print("\n" + "=" * 60)
    print(f"  EXPERIMENTO: {n_runs} RUNS INDEPENDENTES")
    print("=" * 60)

    melhores = []
    for run in range(1, n_runs + 1):
        print(f"\n  Run {run}/{n_runs}...")
        ag = AlgoritmoGenetico(
            tamanho_pop=40,
            num_geracoes=geracoes,
            verbose=verbose_runs,
        )
        melhor = ag.executar()
        melhores.append(melhor.fitness)
        print(f"  Run {run}: melhor fitness = {melhor.fitness:.1f}")

    print(f"\n  Resultados dos {n_runs} runs:")
    print(f"  Média   : {statistics.mean(melhores):.1f}")
    print(f"  Desvio  : {statistics.stdev(melhores):.1f}" if len(melhores) > 1 else "")
    print(f"  Mínimo  : {min(melhores):.1f}")
    print(f"  Máximo  : {max(melhores):.1f}")
    print("=" * 60)


# =============================================================================
# DEMONSTRAÇÃO: Crossover e Mutação
# =============================================================================

def demonstrar_operadores():
    """
    Demonstração pedagógica dos operadores genéticos.
    Mostra como crossover e mutação modificam o material genético.
    """
    print("\n" + "=" * 60)
    print("  DEMONSTRAÇÃO DOS OPERADORES GENÉTICOS")
    print("=" * 60)

    pai1 = Individuo()
    pai2 = Individuo()

    print("\n  PAI 1:", [f"{g:.3f}" for g in pai1.genes])
    print("  PAI 2:", [f"{g:.3f}" for g in pai2.genes])

    f1, f2 = crossover_ponto_unico(pai1, pai2)
    print("\n  [Crossover Ponto Único]")
    print("  Filho 1:", [f"{g:.3f}" for g in f1.genes])
    print("  Filho 2:", [f"{g:.3f}" for g in f2.genes])

    fa, fb = crossover_aritmetico(pai1, pai2)
    print("\n  [Crossover Aritmético]")
    print("  Filho A:", [f"{g:.3f}" for g in fa.genes])
    print("  Filho B:", [f"{g:.3f}" for g in fb.genes])

    mutado = mutar(pai1, taxa=1.0)  # força mutação em todos os genes
    print("\n  [Mutação Forçada de PAI 1]")
    print("  Antes :", [f"{g:.3f}" for g in pai1.genes])
    print("  Depois:", [f"{g:.3f}" for g in mutado.genes])
    deltas = [mutado.genes[i] - pai1.genes[i] for i in range(len(pai1.genes))]
    print("  Delta :", [f"{d:+.3f}" for d in deltas])
    print("=" * 60)


# =============================================================================
# PONTO DE ENTRADA PRINCIPAL
# =============================================================================

def main():
    """
    Ponto de entrada principal do projeto.

    Executa:
      1. Demonstração dos operadores genéticos (pedagógico).
      2. Run principal do AG com exibição completa.
      3. Análise de convergência.
      4. Múltiplos runs para análise de variância (opcional).
    """
    print("\n")
    print("╔══════════════════════════════════════════════════════════╗")
    print("║   AGENTE DINOSSAUROS COM ALGORITMO GENÉTICO              ║")
    print("║   Busca Estocástica e Otimização Evolutiva               ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print()

    random.seed(None)   # seed não-fixada → comportamento genuinamente estocástico

    # ---- Passo 0: Demonstração pedagógica ----
    demonstrar_operadores()
    input("\n  [Enter para iniciar evolução principal...]")

    # ---- Passo 1: Run principal ----
    ag = AlgoritmoGenetico(
        tamanho_pop=TAMANHO_POPULACAO,
        num_geracoes=NUM_GERACOES,
        taxa_crossover=TAXA_CROSSOVER,
        taxa_mutacao=TAXA_MUTACAO,
        elitismo=ELITISMO,
        fitness_alvo=FITNESS_ALVO,
        verbose=True,
        usar_crossover_arit=False,
    )

    melhor = ag.executar()

    # ---- Passo 2: Análise de convergência ----
    analisar_convergencia(ag)

    # ---- Passo 3: Comparar melhor com um indivíduo aleatório ----
    aleatorio = Individuo()
    ag.ambiente.avaliar(aleatorio)
    print("\n  Comparação: Melhor Evoluído vs. Indivíduo Aleatório")
    comparar_individuos(melhor, aleatorio)

    # ---- Passo 4 (opcional): Múltiplos runs ----
    resp = input("\n  Executar análise com múltiplos runs? (s/n): ").strip().lower()
    if resp == "s":
        n = int(input("  Quantos runs? (recomendado: 3-5): ").strip() or "3")
        executar_multiplos_runs(n_runs=n, geracoes=30, verbose_runs=False)

    print("\n  Projeto concluído. Obrigado!\n")


if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════╗
║   AGENTE DINOSSAUROS COM ALGORITMO GENÉTICO              ║
║   Busca Estocástica e Otimização Evolutiva               ║
╚══════════════════════════════════════════════════════════╝


  DEMONSTRAÇÃO DOS OPERADORES GENÉTICOS

  PAI 1: ['2.451', '0.111', '1.535', '0.486', '1.618', '0.401']
  PAI 2: ['0.977', '0.197', '2.383', '0.596', '1.430', '0.345']

  [Crossover Ponto Único]
  Filho 1: ['2.451', '0.111', '1.535', '0.486', '1.430', '0.345']
  Filho 2: ['0.977', '0.197', '2.383', '0.596', '1.618', '0.401']

  [Crossover Aritmético]
  Filho A: ['1.602', '0.161', '2.023', '0.550', '1.509', '0.369']
  Filho B: ['1.826', '0.147', '1.894', '0.533', '1.538', '0.377']

  [Mutação Forçada de PAI 1]
  Antes : ['2.451', '0.111', '1.535', '0.486', '1.618', '0.401']
  Depois: ['2.435', '0.050', '2.124', '0.398', '1.515', '0.298']
  Delta : ['-0.017', '-0.061', '+0.589', '-0.088', '-0.103', '-0.104']

  [Enter para iniciar evolução pri